## Setup inicial

In [1]:
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from utils import logger
from vectorstore import initialize_chroma, genera_y_almacena_embeddings, listar_collections
from carga_de_datos import carga_base_de_conocimientos

load_dotenv()

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("=" * 60)
print("SETUP COMPLETADO")
print("=" * 60)

SETUP COMPLETADO


## Para cada dominio cargamos la bases de conocimiento, generamos embeddings y almacenamos en vector stores, una collection por cada dominio

In [2]:
dominios = ["finanzas", "rrhh", "soporte_it", "legal"]
for dominio in dominios:
    collection = initialize_chroma(dominio)
    chunks = carga_base_de_conocimientos(dominio)
    genera_y_almacena_embeddings(chunks, collection)
    logger.info(
        f"Cargado {len(chunks)} chunks para el dominio {dominio}"
    )
print("vector db cargada")

2026-03-14 15:40:38,719 - Proyecto_M3 - INFO - Collection finanzas created
2026-03-14 15:40:38,722 - Proyecto_M3 - INFO - Base finanzas cargada con 58 chunks
2026-03-14 15:40:39,340 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 15:40:40,081 - Proyecto_M3 - INFO - Cargado 58 chunks para el dominio finanzas
2026-03-14 15:40:40,114 - Proyecto_M3 - INFO - Collection rrhh created
2026-03-14 15:40:40,116 - Proyecto_M3 - INFO - Base rrhh cargada con 59 chunks
2026-03-14 15:40:40,758 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 15:40:41,532 - Proyecto_M3 - INFO - Cargado 59 chunks para el dominio rrhh
2026-03-14 15:40:41,569 - Proyecto_M3 - INFO - Collection soporte_it created
2026-03-14 15:40:41,570 - Proyecto_M3 - INFO - Base soporte_it cargada con 59 chunks
2026-03-14 15:40:42,513 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-1

vector db cargada


In [3]:
# Check collections
listar_collections()



[Collection(name=legal),
 Collection(name=rrhh),
 Collection(name=soporte_it),
 Collection(name=finanzas)]

## Routing

In [ ]:
from agentes.router import route_question
from agentes.constructor_de_agente import build_domain_agent
from langgraph.graph import StateGraph, START, END, MessagesState

hr_agent = build_domain_agent("rrhh")
it_agent= build_domain_agent("soporte_it")
legal_agent = build_domain_agent("legal")
finance_agent = build_domain_agent("finanzas")

def router_node(state):
    question = state["question"]
    route = route_question(question)
    return {"route": route}

def hr_node(state):
    answer = hr_agent.invoke(state["question"])
    return {"answer": answer["result"]}

def it_node(state):
    answer = it_agent.invoke(state["question"])
    return {"answer": answer["result"]}

def legal_node(state):
    answer = legal_agent.invoke(state["question"])
    return {"answer": answer["result"]}

def finance_node(state):
    answer = finance_agent.invoke(state["question"])
    return {"answer": answer["result"]}


builder = StateGraph(dict)

builder.add_node("router", router_node)
builder.add_node("hr", hr_node)
builder.add_node("it", it_node)
builder.add_node("legal", legal_node)
builder.add_node("finance", finance_node)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "HR": "hr",
        "IT": "it",
        "LEGAL": "legal",
        "FINANCE": "finance",
    },
)

for node in ["nolan_specialist", "king_specialist", "davis_specialist", "general"]:
    builder.add_edge(node, END)

graph = builder.compile()
